<a href="https://colab.research.google.com/github/6631501117-art/Room/blob/main/ML_Assignment_Deploy_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#A pre-trained neural network (like MobileNetV2) to perform image classification

#(1) Tools for preparing the deployment ML model with GRADIO

In [ ]:
# Install Gradio for building the web application UI
!pip install gradio -q

In [16]:
#Load data, Train Model, and Predict with Colab Cell
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, decode_predictions, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import io
import pickle

# Load the pre-trained model
model = MobileNetV2(weights="imagenet")

#save the pre-trained model into .pkl
filename = 'model.pkl'
pickle.dump(model, open(filename, 'wb'))

#load model
loaded_model = pickle.load(open('model.pkl', 'rb'))

# File uploader
from google.colab import files

uploaded_file = None # Initialize to None
try:
    # The string argument is the prompt message shown in the dialog
    uploaded_file = files.upload("Upload an image (.jpg, .jpeg, .png)...")
except TypeError as e:
    # Catch the specific TypeError that occurs if 'result' is None internally
    print(f"File upload error: {e}. This often happens if the upload dialog is cancelled or if there's a communication issue.")
    print("Please try uploading the file again.")

if uploaded_file is not None and len(uploaded_file) > 0:
    # Get the first uploaded file's content
    file_name = list(uploaded_file.keys())[0]
    file_content = uploaded_file[file_name]

    # Open the image using BytesIO
    img = Image.open(io.BytesIO(file_content))

    # Preprocess the image
    img = img.resize((224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # Prediction
    preds = loaded_model.predict(x)
    top_preds = decode_predictions(preds, top=3)[0]

    # Display predictions
    print("Predictions:")
    for i, pred in enumerate(top_preds):
        print(f"{i+1}. **{pred[1]}** — {round(pred[2]*100, 2)}%")
else:
    if uploaded_file is not None and len(uploaded_file) == 0:
      print("No file was selected for upload.")
    elif uploaded_file is None:
      print("File upload was cancelled or an error occurred during upload. No file processed.")

Saving d5EZsH8jkoXGFSzBRqQSVc.jpg to Upload an image (.jpg, .jpeg, .png).../d5EZsH8jkoXGFSzBRqQSVc.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predictions:
1. **Pomeranian** — 94.83000183105469%
2. **Samoyed** — 1.159999966621399%
3. **keeshond** — 0.7200000286102295%


In [15]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, decode_predictions, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import gradio as gr
import pickle

# 1. Load the pre-trained model from 'model.pkl'
# The model.pkl was created in the previous cell 2J_oISHtrOIP
model = pickle.load(open('model.pkl', 'rb'))

# 2. Define the core prediction function
# Gradio will automatically pass the uploaded image to this function
def classify_image(img):
    # Resize the image to match MobileNetV2's expected input size
    img = img.resize((224, 224))

    # Preprocess the image
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # Make the prediction
    preds = model.predict(x)
    top_preds = decode_predictions(preds, top=3)[0]

    # Format the results into a dictionary for Gradio's Label component
    # Gradio expects { "Class Name": Probability }
    results = {pred[1]: float(pred[2]) for pred in top_preds}

    return results

# 3. Build the Interactive Gradio Interface
interface = gr.Interface(
    fn=classify_image,                                    # The function to run
    inputs=gr.Image(type="pil", label="Upload an Image"), # Input: Image uploader
    outputs=gr.Label(num_top_classes=3),                  # Output: Confidence bars
    title="📸 MobileNetV2 Image Classifier",
    description="Upload any image (.jpg, .png) and the AI will predict what it is!"
)

# 4. Launch the web app
# share=True creates a public URL you can share instantly
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60c4750ca8d515c754.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#(2) Tools for preparing the deployment ML model on streamlit

In [ ]:
# 1. Install streamlit for building the web application UI
!pip install streamlit -q

In [17]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, decode_predictions, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import pickle

# 1. Setting the web application
st.set_page_config(page_title="Image Classifier", page_icon="📸")
st.title("📸 AI Image Classifier (MobileNetV2)")
st.write("Upload an image (.jpg, .jpeg, .png) and the AI will predict what it is!")

# 2. setting cache
@st.cache_resource
def load_model():
    # Load the pre-trained model from model.pkl
    return pickle.load(open('model.pkl', 'rb'))

model = load_model()

# 3. upload file to streamlit
uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # open and upoad file
    img = Image.open(uploaded_file)
    st.image(img, caption='Uploaded Image', use_container_width=True)

    st.write("🔄 Classifying...")

    # 4. Image preparing
    img_resized = img.resize((224, 224))
    x = image.img_to_array(img_resized)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # 5. Prediction
    preds = model.predict(x)
    top_preds = decode_predictions(preds, top=3)[0]

    # 6. Show the result
    st.subheader("Predictions:")
    for i, pred in enumerate(top_preds):
        class_name = pred[1].replace('_', ' ').title() # setting the font format
        confidence = pred[2] * 100

        # show class
        st.write(f"**{i+1}. {class_name}** — {confidence:.2f}%")
        st.progress(int(confidence))

Overwriting app.py


In [18]:
# 2. Create the requirements.txt file and PLEASE source code HERE !
%%writefile requirements.txt


Overwriting requirements.txt


In [ ]:
# 3. download all file for streamlit and PLEASE source code HERE !
from google.colab import files
